### Week 5 Day 4

AutoGen Core - Distributed

I'm only going to give a Teaser of this!!

Partly because I'm unsure how relevant it is to you. If you'd like me to add more content for this, please do let me know..

In [1]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import Tool
from IPython.display import display, Markdown

from dotenv import load_dotenv

load_dotenv(override=True)

ALL_IN_ONE_WORKER = False

### Start with our Message class

In [2]:

@dataclass
class Message:
    content: str

### And now - a host for our distributed runtime

In [3]:
# grpc is a communication protocol between cross-language calls
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost

host = GrpcWorkerAgentRuntimeHost(address="localhost:50051")
host.start() 

### Let's reintroduce a tool

In [4]:
serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="Useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)

In [5]:
instruction1 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons in favor of choosing AutoGen; the pros of AutoGen."

instruction2 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons against choosing AutoGen; the cons of Autogen."

judge = "You must make a decision on whether to use AutoGen for a project. \
Your research team has come up with the following reasons for and against. \
Based purely on the research from your team, please respond with your decision and brief rationale."

### And make some Agents

In [6]:
class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Judge(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client)
        
    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        message1 = Message(content=instruction1)
        message2 = Message(content=instruction2)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        # calling send message will result in handle_my_message_type of Player1Agent getting called
        # but now it is orchestrated by grpc
        response1 = await self.send_message(message1, inner_1)
        response2 = await self.send_message(message2, inner_2)
        result = f"## Pros of AutoGen:\n{response1.content}\n\n## Cons of AutoGen:\n{response2.content}\n\n"
        judgement = f"{judge}\n{result}Respond with your decision and brief explanation"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + "\n\n## Decision:\n\n" + response.chat_message.content)


In [7]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

if ALL_IN_ONE_WORKER:

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()

    # all agents are registered to the same worker
    await Player1Agent.register(worker, "player1", lambda: Player1Agent("player1"))
    await Player2Agent.register(worker, "player2", lambda: Player2Agent("player2"))
    await Judge.register(worker, "judge", lambda: Judge("judge"))

    agent_id = AgentId("judge", "default")

else:
    # all agents are registered to different workers
    worker1 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker1.start()
    await Player1Agent.register(worker1, "player1", lambda: Player1Agent("player1"))

    worker2 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker2.start()
    await Player2Agent.register(worker2, "player2", lambda: Player2Agent("player2"))

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()
    await Judge.register(worker, "judge", lambda: Judge("judge"))
    agent_id = AgentId("judge", "default")




In [8]:
response = await worker.send_message(Message(content="Go!"), agent_id)

In [9]:
display(Markdown(response.content))

## Pros of AutoGen:
Here are several reasons in favor of choosing AutoGen for your new AI agent project:

1. **Scalability**: AutoGen's modular and extensible framework allows for the creation of scalable systems, enabling you to adapt the architecture as your project grows.

2. **Ease of Use**: It incorporates integrated observability and debugging tools, which simplify the process of monitoring and controlling agent workflows, making development more efficient.

3. **Advanced AI Capabilities**: Designed for complex automation and dynamic environments, AutoGen enhances problem-solving depth and efficiency through sophisticated multi-agent collaboration.

4. **Customization**: The framework offers high levels of customization, allowing developers to tailor solutions to specific needs and requirements, which can improve the relevance and effectiveness of the AI agents.

5. **Collaboration**: AutoGen excels in facilitating seamless collaboration between agents, enabling them to work together autonomously. This can significantly streamline complex tasks and drive more intelligent outcomes.

6. **Optimized Workflows**: It simplifies the orchestration, automation, and optimization of workflows involving large language models (LLMs), enhancing their overall performance and utility in real-world applications.

These advantages make AutoGen a compelling choice for projects requiring advanced AI capabilities and efficient multi-agent system integration.

TERMINATE

## Cons of AutoGen:
Here are some reasons against choosing AutoGen for a new AI Agent project:

1. **Dependency on Input Quality**: The effectiveness of AutoGen heavily relies on the quality of the input data and parameters provided. Poor inputs can lead to suboptimal outputs, necessitating careful selection and validation.

2. **Documentation and Usability Issues**: Users have noted that the documentation for AutoGen can be hard to read and lacks sufficient examples, making it difficult for new users to implement effectively.

3. **Inconsistent Performance**: AutoGen may exhibit inconsistent performance, which could hinder its suitability for production-ready applications. This variability can lead to unpredictable outcomes in real-world applications.

4. **Feedback Loop Limitations**: There can be challenges associated with establishing effective feedback loops, which are crucial for iterating and refining AI models.

5. **Cost and Resource Implications**: Depending on the scale and requirements of the project, the costs associated with using AutoGen (in terms of compute resources, etc.) may not be justified, especially if cheaper alternatives are available.

6. **Potential Quality Reduction in Outputs**: There's concern that widespread use of AutoGen could lead to a decrease in overall quality, particularly in academic or professional contexts, as it might encourage superficial writing.

7. **Compatibility Issues**: AutoGen may face challenges integrating with other open-source models or existing infrastructures, which could complicate implementation.

These factors suggest that while AutoGen has its strengths, there are significant considerations to keep in mind when evaluating its suitability for your project. 

TERMINATE



## Decision:

Based on the provided pros and cons of AutoGen, I recommend **proceeding with AutoGen for the project**. 

**Rationale**: The advantages of scalability, ease of use, advanced AI capabilities, customization, and collaboration make AutoGen a compelling option for projects requiring complex automation and efficient multi-agent interactions. Although there are valid concerns regarding input quality, documentation, and potential performance inconsistencies, the potential benefits outweigh these drawbacks, particularly if careful implementation and monitoring practices are established to mitigate risks.

TERMINATE

In [10]:
await worker.stop()
if not ALL_IN_ONE_WORKER:
    await worker1.stop()
    await worker2.stop()

In [11]:
await host.stop()